## Perform inference on treated single cell profile dataset for hit-calling 
Loads:
- All fold non-shuffled platemapwise models trained. 
- All treated (non-DMSO) failing single cell profiles unseen during training.

Computes inference between all models against all single profiles to yield k-fold split scorings on each single predicting whether it looks more like a cell from control patient with dilated cardiomyopathy or from healthy individual. 

Scores will be used in subsequent analysis assessing candidate treatments.

## Import libraries

In [1]:
import pathlib
import yaml
import json
from joblib import load

import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm

## Pathing and global parameters

In [2]:
random_state = 0
np.random.seed(random_state)
metadata_prefix = "Metadata_"
label_col = "Metadata_cell_type"

# Path to directory with feature selected profiles
path_to_feature_selected_data = pathlib.Path().home() / "mnt" / "bandicoot" /\
    "CFReT_screening_data" / "screen_profiles"
if not path_to_feature_selected_data.exists() and\
    not path_to_feature_selected_data.is_dir():
    raise FileNotFoundError(
        f"Directory {path_to_feature_selected_data} does not exist or is not a directory."
    )

# Find all batch folders
batch_folders = list(path_to_feature_selected_data.glob("batch*"))
batch_folders = [folder for folder in batch_folders if folder.is_dir()]
if not batch_folders:
    raise FileNotFoundError(
        f"No batch folders found in {path_to_feature_selected_data}."
    )

## Other needed paths
fitted_model_dir = pathlib.Path(".") / "models"
if not fitted_model_dir.exists():
    raise FileNotFoundError(f"Fitted model directory not found: {fitted_model_dir}")

datasplit_dir = pathlib.Path(".") / "datasplits"
if not datasplit_dir.exists():
    raise FileNotFoundError(f"Datasplit directory not found: {datasplit_dir}")

encoding_path = datasplit_dir / "cell_type_encoding.json"
if not encoding_path.exists():
    raise FileNotFoundError(f"Cell type encoding file not found: {encoding_path}")

platemap_csv_dir = pathlib.Path(".").resolve().parent.parent  / "metadata" / "updated_platemaps"
if not platemap_csv_dir.exists():
    raise FileNotFoundError(
        f"Directory {platemap_csv_dir} does not exist."
    )

barcode_platemap_file = platemap_csv_dir / "updated_barcode_platemap.csv"
if not barcode_platemap_file.exists():
    raise FileNotFoundError(
        f"Barcode platemap file not found: {barcode_platemap_file}"
    )
platemap_files = [
    file 
    for file in platemap_csv_dir.glob("*.csv")
    if (file.stem not in ["updated_barcode_platemap", "pathways_platemap"])    
]
if not platemap_files:
    raise FileNotFoundError(
        f"No platemap CSV files found in {platemap_csv_dir}."
    )

# output path
output_dir = pathlib.Path(".") / "inference_results"
output_dir.mkdir(exist_ok=True)

## Read in encodings

In [3]:
encoding_dict = json.loads(encoding_path.read_text())
print(f"Loaded cell type encoding for {len(encoding_dict)} cell types.")

Loaded cell type encoding for 2 cell types.


## Read and wrangle platemap metadata
This later helps with collecting treatment wells completely missing single cell profiles,
which may indicate problems with toxicity. 

In [4]:
barcode_platemap_df = pd.read_csv(barcode_platemap_file)

platemap_dfs = []

for platemap_file in platemap_files:

    platemap_repr = platemap_file.stem
    platemap_num = platemap_repr.split("_")[-3]

    df = pd.read_csv(platemap_file)
    df["platemap_file"] = platemap_repr
    df["platemap_num"] = platemap_num
    platemap_dfs.append(df)

platemap_df = pd.concat(platemap_dfs, ignore_index=True)
platemap_df = pd.merge(
    platemap_df, 
    barcode_platemap_df, 
    on=["platemap_file"], 
    how="inner",
    validate="many_to_many"
)

treatment_well_df = platemap_df.loc[
    :,
    ["well_position", "cell_type", "treatment", "platemap_file", "plate_barcode", "platemap_num", "Pathway"]
]
treatment_well_df = treatment_well_df[
    (treatment_well_df["treatment"] != "DMSO") &
    (treatment_well_df["cell_type"] == "failing")    
].rename(columns={
        "plate_barcode": "Metadata_Plate",
        "well_position": "Metadata_Well"
    }
)
treatment_well_df.to_csv(output_dir / "treatment_well.csv")
treatment_well_df.head()

,Metadata_Well,cell_type,treatment,platemap_file,Metadata_Plate,platemap_num,Pathway
4,B03,failing,UCD-0000253,Target_Selective_Library_Screen_Plate_3_with_p...,CARD-CelIns-CX7_251205100001,3,Neuronal Signaling
5,B03,failing,UCD-0000253,Target_Selective_Library_Screen_Plate_3_with_p...,CARD-CelIns-CX7_251203170001,3,Neuronal Signaling
6,B03,failing,UCD-0000253,Target_Selective_Library_Screen_Plate_3_with_p...,CARD-CelIns-CX7_251208160001,3,Neuronal Signaling
7,B03,failing,UCD-0000253,Target_Selective_Library_Screen_Plate_3_with_p...,CARD-CelIns-CX7_251210180001,3,Neuronal Signaling
8,B04,failing,UCD-0000872,Target_Selective_Library_Screen_Plate_3_with_p...,CARD-CelIns-CX7_251205100001,3,Metabolism


## Load all treatment profiles and models and inference

In [5]:
score_dfs = []
cell_counts = []
missing_profile_wells = {}

for batch_folder in batch_folders:

    print(f"Processing batch: {batch_folder.name}")

    platemap_folders = list(
        f 
        for f in batch_folder.glob("*")
        if f.is_dir() 
    )

    for platemap_folder in platemap_folders:

        platemap_repr = platemap_folder.name
        if not platemap_repr.startswith("platemap_"):
            continue
        
        # Resolve fitted model checkpoints
        saved_model_files = (
            fitted_model_dir / platemap_repr
        ).resolve().glob("*original_statsmodels_logit.joblib")
        if not saved_model_files:
            continue
        saved_model_files = sorted(list(saved_model_files))

        # Resolve single cell profile folder
        sc_profile_folder = (platemap_folder / "single_cell_profiles").resolve()
        if not sc_profile_folder.exists() or not sc_profile_folder.is_dir():
            print(f"\tNo single_cell_profiles folder for {platemap_repr}. Skipping.")
            continue

        # Find all feature selected profile files for this platemap
        plate_files = list(
            f
            for f in sc_profile_folder.glob("*_sc_feature_selected.parquet")
            if f.is_file()
        )

        # Inference on each plate file using the fitted models and collect scores and metadata
        for plate_file in plate_files:

            plate_repr = plate_file.stem
            
            # Only inference on non-DMSO treated failing cells
            # Note that training is done on cells that are DMSO treated and both failing and healthy
            df = (
                pl.scan_parquet(plate_file)
                .filter(
                    (pl.col("Metadata_treatment") != "DMSO") & 
                    (pl.col("Metadata_cell_type") == "failing")
                )
                .collect(engine="cpu")
                .to_pandas()
            )
            if df.empty:
                print(f"\t\tNo non DMSO rows for {plate_file.stem}. Skipping.")
                continue
            cell_counts.append(
                df.groupby(['Metadata_treatment', 'Metadata_Well', 'Metadata_Plate']).size().reset_index(name='row_count')
            )

            # Compute missing treatments wells by cross-referencing with the platemap
            plate_name = df['Metadata_Plate'].unique()
            if len(plate_name) != 1:
                raise ValueError(f"Multiple plate names found in {plate_file.stem}: {plate_name}.")            
            plate_name = plate_name[0]
            
            treatment_well_df_subset = treatment_well_df[
                treatment_well_df["Metadata_Plate"] == plate_name
            ].copy()
            treatment_wells = treatment_well_df_subset["Metadata_Well"].unique()
            missing_treatment_wells = set(treatment_wells) - set(df["Metadata_Well"].unique())            
            missing_treatments = treatment_well_df_subset[
                treatment_well_df_subset['Metadata_Well'].isin(missing_treatment_wells)
            ]['treatment'].unique()
            # collect missing profile wells info for this plate
            missing_profile_wells[plate_name] = list(missing_treatments)

            # Iterate over fitted models for this platemap and compute scores
            scores = []
            treatment = df.loc[:, 'Metadata_treatment'].copy()
            for saved_model_file in saved_model_files:

                model = load(saved_model_file)

                feats = list(model.params.index)
                feats = [feat for feat in feats if feat != 'const']
                X = df.loc[:, feats].copy()
                X = sm.add_constant(X, has_constant="add")

                score = model.predict(X)
                if encoding_dict['healthy'] == 0:
                    # convert to larger means more healthy by inverting the scores
                    score_df['mean_score'] = 1 - score_df['mean_score']
                elif encoding_dict['healthy'] == 1:
                    pass
                else:
                    raise ValueError(
                        f"Unexpected encoding for healthy cell type: {encoding_dict['healthy']}"
                    )
                scores.append(score)

            score_df = pd.concat(
                [treatment] + scores,
                axis=1
            )
            score_df.columns = [score_df.columns[0]] + [
                f"fold_{col}_score" for col in score_df.columns[1:]
            ]
            score_df['mean_score'] = score_df.iloc[:, 1:].mean(axis=1)
            score_df['plate'] =  plate_repr
            score_df['platemap'] = platemap_repr
            score_dfs.append(score_df)

Processing batch: batch_1
Processing batch: batch_2
Processing batch: batch_3


## Write results

In [6]:
with (datasplit_dir / "missing_profiles.yaml").open("w") as f:
    yaml.safe_dump(
        missing_profile_wells,
        f,
        default_flow_style=False,
        sort_keys=False,
    )

pd.concat(cell_counts, ignore_index=True).to_csv(
    datasplit_dir / "cell_counts.csv", index=False)

all_score_df = pd.concat(
    score_dfs,
    axis=0,
    ignore_index=True
)
all_score_df.to_parquet(output_dir / 'all_logit_scores.parquet')
all_score_df.head()

,Metadata_treatment,fold_0_score,fold_1_score,fold_2_score,fold_3_score,mean_score,plate,platemap
0,UCD-0000253,0.000014,3.255576e-07,0.000004,0.000004,0.000006,CARD-CelIns-CX7_251210180001_sc_feature_selected,platemap_3
1,UCD-0000253,0.106306,9.999267e-01,0.923445,0.995290,0.756242,CARD-CelIns-CX7_251210180001_sc_feature_selected,platemap_3
2,UCD-0000253,0.000199,1.010449e-01,0.820422,0.009254,0.232730,CARD-CelIns-CX7_251210180001_sc_feature_selected,platemap_3
3,UCD-0000253,0.000109,3.034446e-03,0.001169,0.000344,0.001164,CARD-CelIns-CX7_251210180001_sc_feature_selected,platemap_3
4,UCD-0000253,0.982748,9.973341e-01,0.999946,0.998601,0.994657,CARD-CelIns-CX7_251210180001_sc_feature_selected,platemap_3
